In [ ]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

In [ ]:
!pip install torch_geometric -q

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
from utils.datasets   import NF_IDS_Dataset
from utils.models     import (
    SimpleMLP,
    EdgeGRU_Baseline,        EdgeGRU_Baseline_Entropy,
    StaticGNN_Identity,      StaticGNN_Identity_Entropy,
    ST_GNN_Identity,         ST_GNN_Identity_Entropy,
)
from utils.evaluation import (
    evaluate_test1, evaluate_test2,
    gather_metrics, apply_1sd_rule,
)

# ── Display names ──────────────────────────────────────────────────────────
NAME_MAP = {
    'SimpleMLP_BiasOn':                              'Simple MLP',
    'EdgeGRU_BiasOn':                                'Edge GRU',
    'StaticGNN_BiasOn_robust_Identity':              'Static GNN',
    'ST_GNN_BiasOn_robust_Identity_clone':           'ST-GNN',
    'EdgeGRU_BiasOn_entropy':                        'Edge GRU + Entropy',
    'StaticGNN_BiasOn_robust_Identity_entropy':      'Static GNN + Entropy',
    'ST_GNN_BiasOn_robust_Identity_clone_entropy':   'ST-GNN + Entropy',
}

MODEL_ORDER = [
    'Simple MLP',
    'Edge GRU',           'Edge GRU + Entropy',
    'Static GNN',         'Static GNN + Entropy',
    'ST-GNN',             'ST-GNN + Entropy',
]

COLORS = {
    'Simple MLP':             '#95a5a6',
    'Edge GRU':               '#85c1e9',
    'Edge GRU + Entropy':     '#1a5276',
    'Static GNN':             '#f0b27a',
    'Static GNN + Entropy':   '#7d3c00',
    'ST-GNN':                 '#82e0aa',
    'ST-GNN + Entropy':       '#1e8449',
}

In [ ]:
# ── Shared hyperparameters (same for all 7 models) ─────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NODE_DIM   = 16
EDGE_DIM   = 32
HIDDEN_DIM = 32
DROPOUT    = 0.2
BIAS_VALUE = -2.9968

model_config = {
    'model_name': None,
    'type': None,
    'model_params': {
        'node_dim':          NODE_DIM,
        'edge_dim':          EDGE_DIM,
        'hidden_dim':        HIDDEN_DIM,
        'dropout':           DROPOUT,
        'output_bias_init':  BIAS_VALUE,
    },
    'extra_params': {
        'epochs':        60,
        'batch_steps':   10,
        'pos_weight':    2.0,
        'learning_rate': 0.005,
    }
}

# ── Results directories ────────────────────────────────────────────────────
DIR_NOENT = './results_earlystopping'
DIR_ENT   = './results_earlystopping_entropy'
PLOTS_DIR = './results_comparison/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f'Device: {DEVICE}')

# Test 1 — All Seeds (Mean ± Std)

In [ ]:
# ── Datasets ───────────────────────────────────────────────────────────────
test1_noent = NF_IDS_Dataset(root_dir='./dataset_processed',         split='test')
test1_ent   = NF_IDS_Dataset(root_dir='./dataset_processed_entropy', split='test')

loader_t1_noent = DataLoader(test1_noent, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True)
loader_t1_ent   = DataLoader(test1_ent,   batch_size=1, shuffle=False, num_workers=2, persistent_workers=True)

print(f'Test1 noentropy: {len(test1_noent)} snapshots')
print(f'Test1 entropy:   {len(test1_ent)} snapshots')

In [ ]:
# ── Evaluate all 7 experiments on Test 1 ──────────────────────────────────
T1_DIRNAME = 'test1_results'

# Noentropy
df_t1_mlp    = evaluate_test1(SimpleMLP,           model_config, loader_t1_noent, 'SimpleMLP_BiasOn',                   DEVICE, DIR_NOENT, T1_DIRNAME)
df_t1_egru   = evaluate_test1(EdgeGRU_Baseline,    model_config, loader_t1_noent, 'EdgeGRU_BiasOn',                     DEVICE, DIR_NOENT, T1_DIRNAME)
df_t1_sgnn   = evaluate_test1(StaticGNN_Identity,  model_config, loader_t1_noent, 'StaticGNN_BiasOn_robust_Identity',   DEVICE, DIR_NOENT, T1_DIRNAME)
df_t1_stgnn  = evaluate_test1(ST_GNN_Identity,     model_config, loader_t1_noent, 'ST_GNN_BiasOn_robust_Identity_clone',DEVICE, DIR_NOENT, T1_DIRNAME)

# Entropy
df_t1_egrue  = evaluate_test1(EdgeGRU_Baseline_Entropy,   model_config, loader_t1_ent, 'EdgeGRU_BiasOn_entropy',                      DEVICE, DIR_ENT, T1_DIRNAME)
df_t1_sgnne  = evaluate_test1(StaticGNN_Identity_Entropy, model_config, loader_t1_ent, 'StaticGNN_BiasOn_robust_Identity_entropy',    DEVICE, DIR_ENT, T1_DIRNAME)
df_t1_stgnne = evaluate_test1(ST_GNN_Identity_Entropy,    model_config, loader_t1_ent, 'ST_GNN_BiasOn_robust_Identity_clone_entropy', DEVICE, DIR_ENT, T1_DIRNAME)

In [ ]:
# ── Combine and label ──────────────────────────────────────────────────────
labels_noent = {
    id(df_t1_mlp):   ('Simple MLP',           'noentropy'),
    id(df_t1_egru):  ('Edge GRU',              'noentropy'),
    id(df_t1_sgnn):  ('Static GNN',            'noentropy'),
    id(df_t1_stgnn): ('ST-GNN',                'noentropy'),
    id(df_t1_egrue): ('Edge GRU + Entropy',    'entropy'),
    id(df_t1_sgnne): ('Static GNN + Entropy',  'entropy'),
    id(df_t1_stgnne):('ST-GNN + Entropy',      'entropy'),
}

dfs = [df_t1_mlp, df_t1_egru, df_t1_sgnn, df_t1_stgnn,
       df_t1_egrue, df_t1_sgnne, df_t1_stgnne]

for df in dfs:
    model_name, variant = labels_noent[id(df)]
    df['model']   = model_name
    df['variant'] = variant

df_test1 = pd.concat(dfs, ignore_index=True)
df_test1['model'] = pd.Categorical(df_test1['model'], categories=MODEL_ORDER, ordered=True)
print(f'Total rows: {len(df_test1)}  ({df_test1["model"].nunique()} models × ~5 seeds)')

In [ ]:
# ── Summary table: mean ± std per model ───────────────────────────────────
METRICS = ['Precision', 'Recall', 'F1', 'F2', 'AUC-PR', 'AUC-ROC', 'FPR']
available = [m for m in METRICS if m in df_test1.columns]

rows = []
for model in MODEL_ORDER:
    sub = df_test1[df_test1['model'] == model]
    if sub.empty:
        continue
    row = {'Model': model}
    for m in available:
        mu  = sub[m].mean()
        std = sub[m].std()
        row[m] = f'{mu:.3f} ± {std:.3f}'
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index('Model')
print('\nTest 1 — Mean ± Std across 5 seeds')
print('=' * 90)
display(df_summary)

In [ ]:
# ── Delta table: entropy vs noentropy head-to-head ─────────────────────────
DELTA_METRICS = ['F1', 'AUC-PR', 'Recall', 'FPR']
dm_avail = [m for m in DELTA_METRICS if m in df_test1.columns]

means = df_test1.groupby('model', observed=True)[dm_avail].mean()

arch_pairs = [
    ('Edge GRU',   'Edge GRU + Entropy'),
    ('Static GNN', 'Static GNN + Entropy'),
    ('ST-GNN',     'ST-GNN + Entropy'),
]

delta_rows = []
for noent, ent in arch_pairs:
    if noent not in means.index or ent not in means.index:
        continue
    row = {'Architecture': noent}
    for m in dm_avail:
        v0 = means.loc[noent, m]
        v1 = means.loc[ent,   m]
        d  = v1 - v0
        row[f'{m} (no-ent)'] = f'{v0:.3f}'
        row[f'{m} (ent)']    = f'{v1:.3f}'
        sign = '+' if d >= 0 else ''
        row[f'Δ{m}']         = f'{sign}{d:.3f}'
    delta_rows.append(row)

df_delta = pd.DataFrame(delta_rows).set_index('Architecture')
print('\nEntropy impact by architecture (positive Δ = entropy helps)')
print('=' * 90)
display(df_delta)

In [ ]:
# ── Grouped bar chart: all 7 models × main metrics ────────────────────────
PLOT_METRICS = ['F1', 'AUC-PR', 'Recall', 'Precision', 'FPR']
pm_avail = [m for m in PLOT_METRICS if m in df_test1.columns]

df_melt = df_test1.melt(
    id_vars=['model'], value_vars=pm_avail,
    var_name='Metric', value_name='Value'
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(
    data=df_melt, x='Metric', y='Value', hue='model',
    hue_order=MODEL_ORDER,
    palette=COLORS,
    errorbar='sd', capsize=0.08,
    ax=ax
)
ax.set_title('Test 1 — Model Comparison (mean ± std, 5 seeds)', fontsize=13, weight='bold')
ax.set_ylim(0, 1.05)
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/test1_comparison_barplot.png', dpi=300)
plt.show()

In [ ]:
# ── FPR focused chart ─────────────────────────────────────────────────────
if 'FPR' in df_test1.columns:
    df_fpr = df_test1.groupby('model', observed=True)['FPR'].agg(['mean', 'std']).reset_index()
    df_fpr['model'] = pd.Categorical(df_fpr['model'], categories=MODEL_ORDER, ordered=True)
    df_fpr = df_fpr.sort_values('model')

    fig, ax = plt.subplots(figsize=(9, 4))
    colors_list = [COLORS[m] for m in df_fpr['model']]
    bars = ax.bar(df_fpr['model'], df_fpr['mean'], yerr=df_fpr['std'],
                  color=colors_list, capsize=5, edgecolor='white', linewidth=0.5)
    ax.set_title('Test 1 — False Positive Rate (lower is better)', fontsize=12, weight='bold')
    ax.set_ylabel('FPR')
    ax.set_xlabel('')
    ax.set_xticklabels(df_fpr['model'], rotation=30, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/test1_fpr_comparison.png', dpi=300)
    plt.show()

# Test 2 — Champions only (1-SD Rule)

In [ ]:
# ── Datasets ───────────────────────────────────────────────────────────────
test2_noent = NF_IDS_Dataset(root_dir='./dataset_processed_thu0103',         split='test2')
test2_ent   = NF_IDS_Dataset(root_dir='./dataset_processed_thu0103_entropy', split='test2')

loader_t2_noent = DataLoader(test2_noent, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True)
loader_t2_ent   = DataLoader(test2_ent,   batch_size=1, shuffle=False, num_workers=2, persistent_workers=True)

print(f'Test2 noentropy: {len(test2_noent)} snapshots')
print(f'Test2 entropy:   {len(test2_ent)} snapshots')

In [ ]:
# ── Select champions via 1-SD rule (separately per variant) ───────────────
NAME_MAP_NOENT = {k: v for k, v in NAME_MAP.items() if 'entropy' not in k}
NAME_MAP_ENT   = {k: v for k, v in NAME_MAP.items() if 'entropy' in k}

print('── Noentropy ──')
df_metrics_noent = gather_metrics(os.path.join(DIR_NOENT, 'logs'), NAME_MAP_NOENT)
df_champ_noent   = apply_1sd_rule(df_metrics_noent)

print('\n── Entropy ──')
df_metrics_ent   = gather_metrics(os.path.join(DIR_ENT, 'logs'), NAME_MAP_ENT)
df_champ_ent     = apply_1sd_rule(df_metrics_ent)

In [ ]:
# ── Evaluate champions on Test 2 ──────────────────────────────────────────
T2_DIRNAME = 'test2_results'

noent_pairs = [
    (SimpleMLP,          'SimpleMLP_BiasOn'),
    (EdgeGRU_Baseline,   'EdgeGRU_BiasOn'),
    (StaticGNN_Identity, 'StaticGNN_BiasOn_robust_Identity'),
    (ST_GNN_Identity,    'ST_GNN_BiasOn_robust_Identity_clone'),
]

ent_pairs = [
    (EdgeGRU_Baseline_Entropy,    'EdgeGRU_BiasOn_entropy'),
    (StaticGNN_Identity_Entropy,  'StaticGNN_BiasOn_robust_Identity_entropy'),
    (ST_GNN_Identity_Entropy,     'ST_GNN_BiasOn_robust_Identity_clone_entropy'),
]

dfs_t2_noent = []
for mclass, exp_name in noent_pairs:
    filtered = df_champ_noent[df_champ_noent['Raw_Dir_Name'] == exp_name]
    if filtered.empty:
        print(f'  Warning: no champion found for {exp_name}')
        continue
    df = evaluate_test2(mclass, model_config, loader_t2_noent, exp_name, filtered, DEVICE, DIR_NOENT, T2_DIRNAME, verbose=True)
    if not df.empty:
        dfs_t2_noent.append(df)

dfs_t2_ent = []
for mclass, exp_name in ent_pairs:
    filtered = df_champ_ent[df_champ_ent['Raw_Dir_Name'] == exp_name]
    if filtered.empty:
        print(f'  Warning: no champion found for {exp_name}')
        continue
    df = evaluate_test2(mclass, model_config, loader_t2_ent, exp_name, filtered, DEVICE, DIR_ENT, T2_DIRNAME, verbose=True)
    if not df.empty:
        dfs_t2_ent.append(df)

In [ ]:
# ── Combine champion results ───────────────────────────────────────────────
name_map_by_exp = {
    'SimpleMLP_BiasOn':                              'Simple MLP',
    'EdgeGRU_BiasOn':                                'Edge GRU',
    'StaticGNN_BiasOn_robust_Identity':              'Static GNN',
    'ST_GNN_BiasOn_robust_Identity_clone':           'ST-GNN',
    'EdgeGRU_BiasOn_entropy':                        'Edge GRU + Entropy',
    'StaticGNN_BiasOn_robust_Identity_entropy':      'Static GNN + Entropy',
    'ST_GNN_BiasOn_robust_Identity_clone_entropy':   'ST-GNN + Entropy',
}

def label_df(df, variant):
    df = df.copy()
    # model_name column has format like 'EdgeGRU_BiasOn_seed42' — strip seed suffix
    df['exp_name'] = df['model_name'].str.replace(r'_seed\d+$', '', regex=True)
    df['model']    = df['exp_name'].map(name_map_by_exp).fillna(df['exp_name'])
    df['variant']  = variant
    return df

dfs_t2_labeled = (
    [label_df(d, 'noentropy') for d in dfs_t2_noent] +
    [label_df(d, 'entropy')   for d in dfs_t2_ent]
)

df_test2 = pd.concat(dfs_t2_labeled, ignore_index=True)
df_test2['model'] = pd.Categorical(df_test2['model'], categories=MODEL_ORDER, ordered=True)
print(f'Champions evaluated: {df_test2["model"].nunique()} models')

In [ ]:
# ── Champions summary table ────────────────────────────────────────────────
T2_METRICS = ['Precision', 'Recall', 'F1', 'F2', 'AUC-PR', 'AUC-ROC', 'FPR']
t2_avail = [m for m in T2_METRICS if m in df_test2.columns]

champ_rows = []
for model in MODEL_ORDER:
    sub = df_test2[df_test2['model'] == model]
    if sub.empty:
        continue
    row = {'Model': model}
    if 'seed' in sub.columns:
        row['Seed'] = int(sub['seed'].iloc[0])
    if 'Best_Epoch' in sub.columns:
        row['Best Epoch'] = int(sub['Best_Epoch'].iloc[0])
    for m in t2_avail:
        row[m] = f'{sub[m].iloc[0]:.3f}'
    champ_rows.append(row)

df_champ_summary = pd.DataFrame(champ_rows).set_index('Model')
print('\nTest 2 — Champion per architecture (1-SD Rule)')
print('=' * 90)
display(df_champ_summary)

In [ ]:
# ── Champions bar chart ────────────────────────────────────────────────────
T2_PLOT_METRICS = ['F1', 'AUC-PR', 'Recall', 'Precision', 'FPR']
t2pm_avail = [m for m in T2_PLOT_METRICS if m in df_test2.columns]

# Convert string columns back to float for plotting
df_test2_plot = df_test2.copy()
for m in t2pm_avail:
    df_test2_plot[m] = pd.to_numeric(df_test2_plot[m], errors='coerce')

df_melt2 = df_test2_plot.melt(
    id_vars=['model'], value_vars=t2pm_avail,
    var_name='Metric', value_name='Value'
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(
    data=df_melt2, x='Metric', y='Value', hue='model',
    hue_order=[m for m in MODEL_ORDER if m in df_test2['model'].values],
    palette=COLORS,
    errorbar=None,
    ax=ax
)
ax.set_title('Test 2 — Champion Comparison (1-SD Rule)', fontsize=13, weight='bold')
ax.set_ylim(0, 1.05)
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/test2_champions_barplot.png', dpi=300)
plt.show()